In [1]:
%pip install -q pandas numpy ollama


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Imports

In [2]:
import time

import pandas as pd
from ollama import chat
from pydantic import BaseModel, Field

from src.constants import Column, INPUT_DIR, OUTPUT_DIR

# Config

In [ ]:
MODEL = "llama3:8B"
MAX_REVIEWS = None  # Für den vollständigen Lauf auf None setzen
NUM_CTX = 2048
NUM_PREDICT = 160

## Load dataset

In [4]:
electronics_df = pd.read_csv(INPUT_DIR / "electronics_raw.csv")

if MAX_REVIEWS is not None:
    electronics_df = electronics_df.head(MAX_REVIEWS).copy()
electronics_df = electronics_df.reset_index(drop=True)

result_df = pd.DataFrame()

# Enable LLM

## Setup output schema

In [5]:
class Output(BaseModel):
    subjective_claim_count: int = Field(ge=0)
    objective_claim_count: int = Field(ge=0)
    experiential_detail_claim_count: int = Field(ge=0)
    positive_affect_claim_count: int = Field(ge=0)
    negative_affect_claim_count: int = Field(ge=0)
    uncertain_claim_count: int = Field(ge=0)
    text_sentiment: int = Field(ge=-2, le=2)

In [6]:
MASTER_PROMPT = """
You annotate linguistic properties of online product reviews for an academic
study. Never classify a review as fake, genuine, human-written, or generated.
Evaluate only the supplied review text. Do not infer missing information.

Count claims, not individual words. A sentence can contain more than one claim.
- subjective_claim_count: claims expressing personal preference, emotion,
  judgment, recommendation, or an evaluation that cannot be externally verified.
- objective_claim_count: claims describing externally verifiable specifications,
  events, prices, dimensions, or observable product behaviour.
Do not count a claim in both categories. If no claim is present, return zero for
both counts. First-person pronouns alone do not make a claim subjective.

Independently count claims exhibiting each of the following properties. Unlike
subjective and objective claims, these four categories may overlap with each
other and with either claim type:
- experiential_detail_claim_count: claims containing at least one concrete
  first-hand usage detail. This includes sensory information or specific
  information about locations, times, people, objects, actions, and events in
  the context of product use. A first-person pronoun, generic product description,
  or statement such as 'I used it' is not sufficient without a concrete detail.
- positive_affect_claim_count: claims containing clearly positive emotional
  affect, such as joy, delight, love, excitement, relief, or satisfaction. A
  factual benefit, recommendation, or non-emotional positive evaluation alone
  is not sufficient.
- negative_affect_claim_count: claims containing clearly negative emotional
  affect, such as anger, fear, sadness, disgust, frustration, or disappointment.
  A factual defect or non-emotional negative evaluation alone is not sufficient.
- uncertain_claim_count: claims whose validity, accuracy, or applicability is
  explicitly presented as uncertain or tentative, for example with 'maybe',
  'perhaps', 'probably', 'I think', 'seems', 'might', or 'as far as I can tell'.
  Moderate wording, politeness, personal opinion, and vagueness alone do not
  constitute uncertainty.
Count each claim at most once within each of these four categories. A claim
cannot count as both positive affect and negative affect unless it explicitly
expresses genuinely mixed emotions.

Rate text_sentiment using only the opinion expressed in the text:
-2 clearly negative; -1 somewhat negative; 0 neutral or genuinely mixed;
1 somewhat positive; 2 clearly positive.
Ignore whether claims are true and do not infer a star rating.

Return only valid JSON matching the schema; no explanation or Markdown.
"""

In [7]:
def analyze_review(review_text):
    response = chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": MASTER_PROMPT},
            {"role": "user", "content": f"Review: {review_text}"},
        ],
        keep_alive=-1,
        think=False,
        format=Output.model_json_schema(),
        options={
            "temperature": 0,
            "num_ctx": NUM_CTX,
            "num_predict": NUM_PREDICT,
        },
    )

    return Output.model_validate_json(
        response.message.content
    )

In [8]:
## Test Prompt with one single dataset
test_review = "This is the biggest TV I saw in my entire life. It's absolutely fantastic."
print(analyze_review(test_review))

subjective_claim_count=1 objective_claim_count=1 experiential_detail_claim_count=0 positive_affect_claim_count=1 negative_affect_claim_count=0 uncertain_claim_count=0 text_sentiment=2


## Apply to columns

In [9]:
def analyze_all_reviews():
    total_start = time.perf_counter()
    processing_times = []

    for count, (index, row) in enumerate(electronics_df.iterrows(), start=1):
        review_start = time.perf_counter()

        try:
            result = analyze_review(
                review_text=row[Column.TEXT]
            )

            result_df.loc[index, Column.ID] = row[Column.ID]

            result_df.loc[index, Column.SUBJECTIVE_CLAIM_COUNT] = result.subjective_claim_count
            result_df.loc[index, Column.OBJECTIVE_CLAIM_COUNT] = result.objective_claim_count
            result_df.loc[index, Column.EXPERIENTIAL_DETAIL_CLAIM_COUNT] = result.experiential_detail_claim_count
            result_df.loc[index, Column.POSITIVE_AFFECT_CLAIM_COUNT] = result.positive_affect_claim_count
            result_df.loc[index, Column.NEGATIVE_AFFECT_CLAIM_COUNT] = result.negative_affect_claim_count
            result_df.loc[index, Column.UNCERTAIN_CLAIM_COUNT] = result.uncertain_claim_count
            result_df.loc[index, Column.TEXT_SENTIMENT] = result.text_sentiment

            review_time = time.perf_counter() - review_start
            processing_times.append(review_time)

            if count % 10 == 0:
                avg_time = sum(processing_times) / len(processing_times)
                estimated_remaining = avg_time * (len(electronics_df) - count)

                print(
                    f"{count}/{len(electronics_df)} verarbeitet | "
                    f"Ø {avg_time:.2f} s/Review | "
                    f"Restzeit ca. {estimated_remaining / 60:.1f} min"
                )

        except Exception as e:
            print(f"Fehler bei Zeile {index}: {e}")

    total_time = time.perf_counter() - total_start

    print(f"\nGesamtzeit: {total_time / 60:.2f} Minuten")

    if processing_times:
        avg_time = sum(processing_times) / len(processing_times)
        print(f"Durchschnittliche Zeit pro erfolgreichem Review: {avg_time:.2f} Sekunden")


analyze_all_reviews()

10/3986 verarbeitet | Ø 7.30 s/Review | Restzeit ca. 483.5 min
20/3986 verarbeitet | Ø 7.21 s/Review | Restzeit ca. 476.9 min
30/3986 verarbeitet | Ø 7.16 s/Review | Restzeit ca. 472.3 min
40/3986 verarbeitet | Ø 7.27 s/Review | Restzeit ca. 478.0 min
50/3986 verarbeitet | Ø 7.24 s/Review | Restzeit ca. 475.0 min
60/3986 verarbeitet | Ø 7.21 s/Review | Restzeit ca. 471.6 min
70/3986 verarbeitet | Ø 7.18 s/Review | Restzeit ca. 468.5 min
80/3986 verarbeitet | Ø 7.15 s/Review | Restzeit ca. 465.7 min
90/3986 verarbeitet | Ø 7.14 s/Review | Restzeit ca. 463.4 min
100/3986 verarbeitet | Ø 7.12 s/Review | Restzeit ca. 460.9 min
110/3986 verarbeitet | Ø 7.12 s/Review | Restzeit ca. 460.0 min
120/3986 verarbeitet | Ø 7.10 s/Review | Restzeit ca. 457.6 min
130/3986 verarbeitet | Ø 7.10 s/Review | Restzeit ca. 456.0 min
140/3986 verarbeitet | Ø 7.09 s/Review | Restzeit ca. 454.6 min
150/3986 verarbeitet | Ø 7.08 s/Review | Restzeit ca. 452.8 min
160/3986 verarbeitet | Ø 7.08 s/Review | Restzeit

# Save to CSV

In [10]:
result_df.to_csv(
    OUTPUT_DIR / "llm" / f"{MODEL}_results.csv",
    index=False, 
    encoding="utf-8"
)